# Set up Google Colab and Link Google Drive

In [ ]:
import os
from google.colab import drive

# 1. Mount the full drive
drive.mount('/content/drive')

# 2. Change directory to your project folder
os.chdir('/content/drive/My Drive/agentic_coding_demo/')
print('Current working directory:', os.getcwd())

# Load jax
import jax.tools.colab_tpu
import jax
from jax.extend import backend
print('JAX platform:', backend.get_backend().platform)
print('JAX devices:', jax.devices())

# 1. Install Flax and Tensorflow (for data)

In [ ]:
!pip install -U "jax[tpu]"
!pip install -U flax

In [ ]:
!pip install tensorflow

# 2. Load the MNIST Dataset

`MNISTDataLoader` handles downloading, normalizing to `[0, 1]`, shuffling, and batching — all in one call.

In [ ]:
from custom_utils import MNISTDataLoader

loader = MNISTDataLoader(batch_size=32, train_steps=1200)
train_ds, test_ds = loader.load()

# 3. Define the Model

`CNN` is a Flax NNX module: two conv blocks (Conv → BatchNorm → Dropout → AvgPool) followed by two linear layers.

In [ ]:
from flax import nnx
from custom_utils import CNN

model = CNN(rngs=nnx.Rngs(0))
nnx.display(model)

# Run the Model on a Dummy Input

In [ ]:
import jax.numpy as jnp

y = model(jnp.ones((1, 28, 28, 1)), nnx.Rngs(0))
y

# 4. Train the Model

`Trainer` owns the optimizer (`adamw`), metrics (`accuracy` + `loss`), and the training loop. It evaluates on the test set every `eval_every` steps and plots live learning curves.

In [ ]:
from custom_utils import Trainer

trainer = Trainer(model, learning_rate=0.005, momentum=0.9)
trainer.train(train_ds, test_ds, train_steps=1200, eval_every=200)

# 5. Perform Inference on the Test Set

`Predictor` switches the model to eval mode and provides `predict` and `visualize_predictions` methods.

In [ ]:
from custom_utils import Predictor

predictor = Predictor(model)

test_batch = test_ds.as_numpy_iterator().next()
predictor.visualize_predictions(test_batch, grid_shape=(5, 5))

# 6. Export the Model

`ModelExporter` saves the trained model as a TF SavedModel via Orbax.

In [ ]:
from custom_utils import ModelExporter

exporter = ModelExporter(model)
output_dir = exporter.export('/tmp/mnist_export')

In [ ]:
!ls /tmp/mnist_export

# 7. Update Google Drive

In [ ]:
drive.flush_and_unmount()
print('All changes made in this colab session should now be visible in Drive.')